# Classical Sentiment Classification Baselines

This notebook repeats the best performing classical NLP experiments from the legacy notebook on the cleaned project splits.

Models:
1. Bag-of-Words + Logistic Regression
2. TF-IDF + Naive Bayes
3. TF-IDF + Logistic Regression
4. TF-IDF + Linear SVC




## Choice of Classical Baselines

The classical experiments focus on linear models trained on Bag-of-Words and TF-IDF representations. This is a natural starting point for sentiment analysis because these representations produce high-dimensional and sparse feature vectors, where linear models often perform very well.

Logistic Regression is used as the main classical baseline because it is simple, fast, robust, and well suited for sparse text features. It also provides a strong comparison point before moving to transformer-based models.

Naive Bayes is included as a traditional probabilistic baseline for text classification. Although it relies on stronger independence assumptions, it is efficient and often competitive on sentiment classification tasks.

Linear SVC is included as another strong linear classifier for sparse TF-IDF features.

More complex tree-based models, such as Random Forests or Gradient Boosting methods, are not included at this stage. These models are usually less effective on sparse, high-dimensional TF-IDF features and can add computational complexity without providing a clear advantage. For this reason, the classical baseline section focuses on methods that are both standard and well matched to the feature representation.

## Imports

In [1]:
import re
from pathlib import Path

import pandas as pd
import spacy
from bs4 import BeautifulSoup

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

## Load the processed dataset

In [2]:
DATA_DIR = Path("../data/processed")

train_df = pd.read_csv(DATA_DIR / "train.csv")
valid_df = pd.read_csv(DATA_DIR / "valid.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

train_df.head()

,text,label
0,I was seriously looking forward to seeing this...,0
1,"`Mad Dog' Earle is back, along with his sad-sa...",1
2,"* Firstly, although many say it is the worst o...",1
3,"I love science fiction, I am fascinated by Egy...",0
4,One measurement for the greatness of a movie i...,1


In [3]:
print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

print("\nTrain label distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation label distribution:")
print(valid_df["label"].value_counts().sort_index())

print("\nTest label distribution:")
print(test_df["label"].value_counts().sort_index())

Train: (8000, 2)
Validation: (2000, 2)
Test: (2000, 2)

Train label distribution:
label
0    4000
1    4000
Name: count, dtype: int64

Validation label distribution:
label
0    1000
1    1000
Name: count, dtype: int64

Test label distribution:
label
0    1000
1    1000
Name: count, dtype: int64


## Preprocessing

The preprocessing follows the legacy notebook closely:
- remove HTML tags
- lowercase text
- tokenize with spaCy
- remove punctuation and spaces
- remove stopwords, but keep negations
- lemmatize tokens


In [4]:
nlp = spacy.load("en_core_web_sm")

negations = {
    "not", "no", "n't", "isn't", "aren't", "won't",
    "can't", "never", "nothing"
}


def preprocess_review(review: str) -> str:
    # Remove HTML tags
    review = BeautifulSoup(review, "html.parser").get_text(" ")

    # Remove social-style mentions/hashtags if present
    review = re.sub(r"(@[A-Za-z0-9_]+)|(#\S+)", " ", review)

    doc = nlp(review.lower())

    clean_tokens = []
    for token in doc:
        if token.is_punct or token.is_space:
            continue
        if token.is_stop and token.text not in negations:
            continue

        clean_tokens.append(token.lemma_)

    return " ".join(clean_tokens)

In [5]:
# Quick preprocessing check
example = train_df.loc[0, "text"]

print("Original review:")
print(example[:500])

print("\nProcessed review:")
print(preprocess_review(example)[:500])

Original review:
I was seriously looking forward to seeing this film because it seemed truly promising from the coming attractions: Jim Carrey with Godlike powers was an idea that most definitely worked for me. As a huge fan, I was sure he'd be supremely in his element with such a promising premise, and what could go wrong? Yesterday, my bubble got burst big-time, boys and girls, because I saw the movie.   The first act (where it's set up that he hates his life, he's a disgruntled employee and a majorly unhappy 

Processed review:
seriously look forward see film truly promising come attraction jim carrey godlike power idea definitely work huge fan sure supremely element promising premise wrong yesterday bubble get burst big time boy girl see movie act set hat life disgruntled employee majorly unhappy camper ax grind god serviceable second act summon god telephone receive power almighty great carrey get fun new toy pleasure watch funny act wretche belief rot start set dinner scene bruce

In [6]:
# Preprocess all splits
train_df["processed_text"] = train_df["text"].apply(preprocess_review)
valid_df["processed_text"] = valid_df["text"].apply(preprocess_review)
test_df["processed_text"] = test_df["text"].apply(preprocess_review)

train_df[["text", "processed_text", "label"]].head()

,text,processed_text,label
0,I was seriously looking forward to seeing this...,seriously look forward see film truly promisin...,0
1,"`Mad Dog' Earle is back, along with his sad-sa...",` mad dog earle sad sack moll marie fickle clu...,1
2,"* Firstly, although many say it is the worst o...",firstly bad series not think true consider ide...,1
3,"I love science fiction, I am fascinated by Egy...",love science fiction fascinate egyptian mythol...,0
4,One measurement for the greatness of a movie i...,measurement greatness movie come t.v right wan...,1


## Evaluation helper

In [7]:
target_names = ["negative", "positive"]


def evaluate_model(model_name, model, X_train, y_train, X_valid, y_valid):
    model.fit(X_train, y_train)

    y_pred = model.predict(X_valid)

    accuracy = accuracy_score(y_valid, y_pred)

    print(f"{model_name}")
    print("=" * len(model_name))
    print(f"Accuracy: {accuracy:.4f}")
    print("\nClassification report:")
    print(classification_report(y_valid, y_pred, target_names=target_names))

    return {
        "model_name": model_name,
        "model": model,
        "accuracy": accuracy,
        "predictions": y_pred,
    }

In [8]:
X_train = train_df["processed_text"]
y_train = train_df["label"]

X_valid = valid_df["processed_text"]
y_valid = valid_df["label"]

X_test = test_df["processed_text"]
y_test = test_df["label"]

## 1. Bag-of-Words + Logistic Regression

In [9]:
bow_lr_pipeline = Pipeline([
    ("vectorizer", CountVectorizer(max_features=5000, ngram_range=(1, 2))),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

bow_lr_results = evaluate_model(
    "Bag-of-Words + Logistic Regression",
    bow_lr_pipeline,
    X_train,
    y_train,
    X_valid,
    y_valid,
)

Bag-of-Words + Logistic Regression
Accuracy: 0.8445

Classification report:
              precision    recall  f1-score   support

    negative       0.85      0.84      0.84      1000
    positive       0.84      0.85      0.85      1000

    accuracy                           0.84      2000
   macro avg       0.84      0.84      0.84      2000
weighted avg       0.84      0.84      0.84      2000



## 2. TF-IDF + Naive Bayes

In [10]:
tfidf_nb_pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ("classifier", MultinomialNB()),
])

tfidf_nb_results = evaluate_model(
    "TF-IDF + Naive Bayes",
    tfidf_nb_pipeline,
    X_train,
    y_train,
    X_valid,
    y_valid,
)

TF-IDF + Naive Bayes
Accuracy: 0.8515

Classification report:
              precision    recall  f1-score   support

    negative       0.87      0.83      0.85      1000
    positive       0.84      0.88      0.85      1000

    accuracy                           0.85      2000
   macro avg       0.85      0.85      0.85      2000
weighted avg       0.85      0.85      0.85      2000



## 3. TF-IDF + Logistic Regression

In [11]:
tfidf_lr_pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

tfidf_lr_results = evaluate_model(
    "TF-IDF + Logistic Regression",
    tfidf_lr_pipeline,
    X_train,
    y_train,
    X_valid,
    y_valid,
)

TF-IDF + Logistic Regression
Accuracy: 0.8670

Classification report:
              precision    recall  f1-score   support

    negative       0.89      0.84      0.86      1000
    positive       0.85      0.90      0.87      1000

    accuracy                           0.87      2000
   macro avg       0.87      0.87      0.87      2000
weighted avg       0.87      0.87      0.87      2000



In [26]:
from sklearn.svm import LinearSVC

tfidf_svm_model = Pipeline([
    ("vectorizer", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ("classifier", LinearSVC(C=1.0)),
])

tfidf_svm_model.fit(train_df["processed_text"], train_df["label"])

tfidf_svm_predictions = tfidf_svm_model.predict(valid_df["processed_text"])

tfidf_svm_accuracy = accuracy_score(valid_df["label"], tfidf_svm_predictions)

print("Accuracy:", tfidf_svm_accuracy)
print("\nClassification Report:\n")
print(classification_report(
    valid_df["label"],
    tfidf_svm_predictions,
    target_names=["negative", "positive"]
))

tfidf_svm_results = {
    "model_name": "TF-IDF + Linear SVC",
    "model": tfidf_svm_model,
    "accuracy": tfidf_svm_accuracy,
    "predictions": tfidf_svm_predictions,
}

Accuracy: 0.858

Classification Report:

              precision    recall  f1-score   support

    negative       0.87      0.84      0.86      1000
    positive       0.85      0.87      0.86      1000

    accuracy                           0.86      2000
   macro avg       0.86      0.86      0.86      2000
weighted avg       0.86      0.86      0.86      2000



## Compare validation results

In [21]:
results = [
    bow_lr_results,
    tfidf_nb_results,
    tfidf_lr_results,
    tfidf_svm_results,
]

comparison_df = pd.DataFrame([
    {
        "model": result["model_name"],
        "validation_accuracy": round(result["accuracy"], 4),
    }
    for result in results
])

comparison_df.sort_values("validation_accuracy", ascending=False)

,model,validation_accuracy
2,TF-IDF + Logistic Regression,0.8670
3,TF-IDF + Linear SVC,0.8580
1,TF-IDF + Naive Bayes,0.8515
0,Bag-of-Words + Logistic Regression,0.8445


## Error examples

In [22]:
# Inspect misclassified validation examples

label_map = {
    0: "negative",
    1: "positive",
}

valid_errors = valid_df.copy()
valid_errors["predicted_label"] = tfidf_lr_results["predictions"]
valid_errors["true_sentiment"] = valid_errors["label"].map(label_map)
valid_errors["predicted_sentiment"] = valid_errors["predicted_label"].map(label_map)

misclassified = valid_errors[
    valid_errors["label"] != valid_errors["predicted_label"]
].copy()

print(f"Number of misclassified examples: {len(misclassified)}")
print(f"Validation set size: {len(valid_errors)}")
print(f"Error rate: {len(misclassified) / len(valid_errors):.3f}")

Number of misclassified examples: 266
Validation set size: 2000
Error rate: 0.133


In [23]:
# Display a few misclassified examples clearly

pd.set_option("display.max_colwidth", 500)

examples_to_show = misclassified[
    ["true_sentiment", "predicted_sentiment", "text"]
].sample(10, random_state=42)

for i, row in examples_to_show.reset_index(drop=True).iterrows():
    print("=" * 100)
    print(f"Example {i + 1}")
    print(f"True label:      {row['true_sentiment']}")
    print(f"Predicted label: {row['predicted_sentiment']}")
    print("-" * 100)
    print(row["text"][:1500])
    print()

Example 1
True label:      negative
Predicted label: positive
----------------------------------------------------------------------------------------------------
The plot of 'House of Games' is the strongest thing about it: a successful author and psychologist is conned by a gang of grifters, but in discovering the wicked part of herself that enjoys the thrill of what they do, she finally gets her revenge. That's about the pitch: but someone has to take responsibility for it coming across as being acted by puppets. It has to be the director Mamet: Lindsay Crouse has had a varied and pretty steady TV and film career, so she can't perform this badly all the time. She's supposed to go from uptight, cool, controlled professional to calculating, wicked fast lady having fun, as shown by the change from beige trouser suit (which she seems to wear for three days straight, including underwear) to floppy floral sundress. But everyone seems to be speaking their lines the same clipped, precise wa

The misclassified examples often contain mixed sentiment, contrastive phrasing, or a delayed final judgment. Negative reviews may include positive descriptions of specific aspects before ending with an overall negative evaluation, while positive reviews may begin with negative expectations before reversing them. This suggests that simple BoW/TF-IDF models are limited by their weak treatment of context and discourse structure.

## Evaluate the best model on the test set

In [ ]:
best_result = max(results, key=lambda result: result["accuracy"])
best_model = best_result["model"]
best_model_name = best_result["model_name"]

test_pred = best_model.predict(X_test)

print(f"Best validation model: {best_model_name}")
print(f"Test accuracy: {accuracy_score(y_test, test_pred):.4f}")

print("\nTest classification report:")
print(classification_report(y_test, test_pred, target_names=target_names))

Best validation model: TF-IDF + Logistic Regression
Test accuracy: 0.8710

Test classification report:
              precision    recall  f1-score   support

    negative       0.89      0.85      0.87      1000
    positive       0.85      0.89      0.87      1000

    accuracy                           0.87      2000
   macro avg       0.87      0.87      0.87      2000
weighted avg       0.87      0.87      0.87      2000



## Notes

The best validation result among the classical baselines is obtained by TF-IDF with Logistic Regression. TF-IDF with Linear SVC is the second strongest model in this comparison, reaching 0.858 validation accuracy. Logistic Regression will be used as the main classical reference model for later transformer-based experiments.
